In [50]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [51]:
Data = pd.read_csv('../data/processed_data.csv')
df = pd.DataFrame(Data)

In [56]:
def Logreg(X, y, Testsize, solvers=['liblinear', 'lbfgs', 'newton-cg', 'sag', 'saga']):
    eval_list = []
    y_pred = None

    for x in Testsize:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=x, random_state=0
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        for solver_name in solvers:
            try:
                logreg = LogisticRegression(
                    solver=solver_name,
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=0
                )
                logreg.fit(X_train_scaled, y_train)

                y_pred = logreg.predict(X_test_scaled)

                acc = metrics.accuracy_score(y_test, y_pred)
                score = logreg.score(X_test_scaled, y_test)

                eval_list.append({
                    'Test_size': x,
                    'Solver': solver_name,
                    'acc': acc,
                    'score': score
                })
            except Exception as e:
                print(f"Solver {solver_name} with test_size {x} failed: {e}")

    df_evaluation = pd.DataFrame(eval_list)

    if not df_evaluation.empty:
        df_evaluation = df_evaluation.set_index(['Test_size', 'Solver'])

    return X_train, X_test, y_train, y_test, y_pred, df_evaluation

In [62]:
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #90ee90; color: black; font-weight: bold;' if v else '' for v in is_max]

In [58]:
X = df.drop(columns=['Personal Loan', 'Place', 'ID'], errors='ignore')
X = pd.get_dummies(X, drop_first=True)
y = df['Personal Loan']

In [64]:
X_train, X_test, y_train, y_test, y_pred, df_eval = Logreg(X, y, [0.1, 0.15, 0.2, 0.25, 0.3, 0.35])

df_eval.style.apply(highlight_max, subset=['acc', 'score'])